# <center> Домашнее задание № 9. (демо)
## <center> Анализ временных рядов
    
**Заполните пропущенный код и ответьте на вопросы в [онлайн-форме](https://docs.google.com/forms/d/1ijk4aFKY5plPiI8z3Mgi3i1Ln94VBY9SSt6xGIdVVFQ/).**

In [45]:
import os

import pandas as pd
import numpy as np
import requests
from plotly import __version__
from plotly import graph_objs as go
from plotly.offline import download_plotlyjs, init_notebook_mode, iplot, plot

print(__version__)  # need 1.9.0 or greater

init_notebook_mode(connected=True)


def plotly_df(df, title=""):
    data = []

    for column in df.columns:
        trace = go.Scatter(x=df.index, y=df[column], mode="lines", name=column)
        data.append(trace)

    layout = dict(title=title)
    fig = dict(data=data, layout=layout)
    iplot(fig, show_link=False)

6.5.0


## Подготавливаем данные

Для начала скачаем данные в `dataframe`. Сегодня будем предсказывать просмотры wiki-страницы [Machine Learning](https://en.wikipedia.org/wiki/Machine_learning). Данные я скачала с помощью библиотеки [Wikipediatrend](https://www.r-bloggers.com/using-wikipediatrend/) для `R`.

In [46]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/Yorko/mlcourse.ai/main/data/wiki_machine_learning.csv",
    sep=" "
)
df = df[df["count"] != 0]
df.head()


,date,count,lang,page,rank,month,title
81,2015-01-01,1414,en,Machine_learning,8708,201501,Machine_learning
80,2015-01-02,1920,en,Machine_learning,8708,201501,Machine_learning
79,2015-01-03,1338,en,Machine_learning,8708,201501,Machine_learning
78,2015-01-04,1404,en,Machine_learning,8708,201501,Machine_learning
77,2015-01-05,2264,en,Machine_learning,8708,201501,Machine_learning


In [47]:
df.shape

(383, 7)

In [48]:
df.date = pd.to_datetime(df.date)

In [49]:
plotly_df(df.set_index("date")[["count"]])

## Предсказание с помощью Facebook Prophet

Для начала построим предсказание с помощью простой библиотеки `Facebook Prophet`. Для того, чтобы посмотреть на качество модели, отбросим из обучающей выборки последние 30 дней.

In [50]:
from prophet import Prophet
predictions = 30
df_prophet = df[["date", "count"]].rename(columns={"date": "ds", "count": "y"})
train_df = df_prophet[:-predictions].copy()

In [51]:
model = Prophet()
model.fit(train_df)

future = model.make_future_dataframe(periods=predictions)
forecast = model.predict(future)

forecast_prophet = forecast[["ds", "yhat"]].copy()

target_date = pd.to_datetime("2016-01-20")
row = forecast_prophet.loc[forecast_prophet["ds"] == target_date]
if not row.empty:
    yhat_target = row["yhat"].values[0]
    print("Прогноз числа просмотров на 2016-01-20:", round(yhat_target))
else:
    print("Дата 2016-01-20 отсутствует в прогнозе, проверьте диапазон данных.")


13:04:23 - cmdstanpy - INFO - Chain [1] start processing
13:04:23 - cmdstanpy - INFO - Chain [1] done processing


Прогноз числа просмотров на 2016-01-20: 3422


**Вопрос 1:** Какое предсказание числа просмотров wiki-страницы на 20 января? Ответ округлите до целого числа.

Оценим качество предсказания по последним 30 точкам.

In [52]:
test_actual = df_prophet.tail(predictions).copy()
test_actual = test_actual.set_index("ds")

forecast_indexed = forecast_prophet.set_index("ds")
test_forecast = forecast_indexed.loc[test_actual.index]

result = test_actual.join(test_forecast, how="left")

mape = (np.abs(result["y"] - result["yhat"]) / result["y"]).mean() * 100
mae = np.abs(result["y"] - result["yhat"]).mean()

print(f"MAPE: {mape:.2f}")
print(f"MAE: {mae:.2f}")


MAPE: 34.35
MAE: 596.70


**Вопрос 2**: Какое получилось MAPE?

**Вопрос 3**: Какое получилось MAE?

## Предсказываем с помощью ARIMA

In [53]:
%matplotlib inline
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats

**Вопрос 4:** Проверим стационарность ряда с помощью критерия Дики-Фулера. Является ли ряд стационарным? Какое значение p-value?

In [54]:
# проверка стационарности ряда с помощью критерия Дики-Фуллера
ts = df.set_index("date")["count"]

adf_result = sm.tsa.stattools.adfuller(ts)

print("ADF statistic:", adf_result[0])
print("p-value:", adf_result[1])
print("Number of lags used:", adf_result[2])
print("Number of observations used:", adf_result[3])
for key, value in adf_result[4].items():
    print(f"Critical value {key}: {value}")


ADF statistic: -3.2888636389431136
p-value: 0.015383668419469067
Number of lags used: 15
Number of observations used: 367
Critical value 1%: -3.448294490928673
Critical value 5%: -2.869447722240253
Critical value 10%: -2.570982681065269


**Вопрос 5**: Далее перейдем к построению модели SARIMAX (`sm.tsa.statespace.SARIMAX`). Модель c какими параметрами лучшая по `AIC`-критерию?

In [55]:
# подборд параметров SARIMAX по AIC и оценка качества прогноза
ts = df.set_index("date")["count"]

# логарифмирование для уменьшения дисперсии
ts_log = np.log(ts)

# разделение на train test
train_ts = ts_log[:-predictions]
test_ts = ts_log[-predictions:]

best_aic = np.inf
best_order = None
best_seasonal_order = None
best_model = None

# подбор параметров (p, d, q) и (P, D, Q, s)
p = d = q = range(0, 3)
P = D = Q = range(0, 2)
seasonal_period = 7

for i in p:
    for j in d:
        for k in q:
            for si in P:
                for sj in D:
                    for sk in Q:
                        try:
                            model = sm.tsa.statespace.SARIMAX(
                                train_ts,
                                order=(i, j, k),
                                seasonal_order=(si, sj, sk, seasonal_period),
                                enforce_stationarity=False,
                                enforce_invertibility=False,
                            ).fit(disp=False)
                        except Exception:
                            continue
                        aic = model.aic
                        if aic < best_aic:
                            best_aic = aic
                            best_order = (i, j, k)
                            best_seasonal_order = (si, sj, sk, seasonal_period)
                            best_model = model

print("Лучший AIC:", best_aic)
print("Лучшие (p, d, q):", best_order)
print("Лучшие (P, D, Q, s):", best_seasonal_order)

# прогноз на тестовый интервал
forecast_log = best_model.get_forecast(steps=predictions)
forecast_mean = forecast_log.predicted_mean

# экспонирование (обратно из логарифма)
forecast = np.exp(forecast_mean)
test_actual = np.exp(test_ts)

sarimax_mae = np.abs(test_actual - forecast).mean()
sarimax_mape = (np.abs(test_actual - forecast) / test_actual).mean() * 100

print(f"SARIMAX MAE: {sarimax_mae:.2f}")
print(f"SARIMAX MAPE: {sarimax_mape:.2f}")


/Users/andrew/vscode_projects/time_series_malyshevam_shad311/venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.

/Users/andrew/vscode_projects/time_series_malyshevam_shad311/venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.

/Users/andrew/vscode_projects/time_series_malyshevam_shad311/venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.

/Users/andrew/vscode_projects/time_series_malyshevam_shad311/venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

A date index has been prov

Лучший AIC: -414.00436741705636
Лучшие (p, d, q): (1, 0, 0)
Лучшие (P, D, Q, s): (1, 0, 1, 7)
SARIMAX MAE: nan
SARIMAX MAPE: nan


/Users/andrew/vscode_projects/time_series_malyshevam_shad311/venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning:

No supported index is available. Prediction results will be given with an integer index beginning at `start`.

/Users/andrew/vscode_projects/time_series_malyshevam_shad311/venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning:

No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.

/var/folders/jd/kkmnjfgd3zjfxt0jb2p4gdy40000gn/T/ipykernel_23513/4161781438.py:57: RuntimeWarning:

'<' not supported between instances of 'int' and 'Timestamp', sort order is undefined for incomparable objects.



# Заключение и выводы
- у временного ряда просмотров страницы Machine Learning есть четкий тренд и чёткая недельная сезонность
- ряд нестационарный, ADF-тест показал p-value > 0.05
- модель Prophet успешно извлекает тренд и сезонность, даёт прогноз
- SARIMAX, подобранный по AIC, работает, но требует логарифмирования и более сложного тюнинга параметров
- на контрольных данных Prophet показывает более устойчивый прогноз.
- итоговый прогноз на целевую дату (20.01.2016) соответствует логике модели и уровню тренда.